In [ ]:
%matplotlib qt5
import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.vector import Miller, Vector3d
from orix.plot import IPFColorKeyTSL
from matplotlib.lines import Line2D
from dataclasses import dataclass

import plotly.graph_objects as go

In [ ]:
PATH_mini = Path("D:/Datasets/Example_set/mini_centered.hspy")
PATH_cropped = Path("D:/Datasets/Example_set/cropped_Ag_centered.hspy")
Path_full = Path("D:/Datasets/Example_set/SPED_Ag_full_centered.hspy")

PATH_test = Path("D:/Datasets/Example_set/test.zspy")

colors = ["blue", "red", "green", "orange"]
labels = ["SPED5", "SPED6", "SPED7", "SPED8"]

In [ ]:
ploter = hs.load(PATH_cropped, lazy=True)
ploter.plot()

In [ ]:
# Define the crystal structure for silver (Ag)
a = 4.08
atoms = [Atom('Ag', [0, 0, 0]), Atom('Ag', [0.5, 0.5, 0]), Atom('Ag', [0.5, 0, 0.5]), Atom('Ag', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Ag', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [ ]:
def function1(path, masking=False):
    SPED_data = hs.load(path, lazy=True)
    SPED_data.change_dtype("float32")

    ny, nx = SPED_data.axes_manager.signal_shape
    center = (ny/2 - 0.5, nx/2 - 0.5)
    SPED_data.calibration(center=center)
    if masking == True:
        SPED_template = SPED_data.template_match_disk(disk_r=3, subtract_min=False)
        Peaks_data = SPED_template.get_diffraction_vectors(min_distance=3, threshold_abs=0.6)
        SPED_mask = Peaks_data.to_mask(disk_r=3)
        return SPED_data * SPED_mask
    else:
        return SPED_data

def function2(data):
    SPED_azi = data.get_azimuthal_integral2d(npt=256)
    results = SPED_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    results.compute()
    
    single_phase = results.to_single_phase_orientations()[:,:,0]
    xmap = results.to_crystal_map()
    oris = xmap.orientations
    corrs = results.data[:,:,0,1].flatten()
    
    return results, xmap, oris, corrs, single_phase

@dataclass
class SPEDResult:
    results: object
    xmap: object
    oris: object
    corrs: object
    single_phase: object   

    def colormap(self, xmap, oris, corrs):
        oris_z = key_z.orientation2color(self.oris)
        xmap.plot(oris_z, overlay=self.corrs, remove_padding=True)
        oris_x = key_x.orientation2color(oris)
        xmap.plot(oris_x, overlay=self.corrs, remove_padding=True)
        oris_y = key_y.orientation2color(self.oris)
        self.xmap.plot(oris_y, overlay=self.corrs, remove_padding=True)

    def ipf_correlation(self, ix, iy):
        correlations_i = self.results.inav[ix, iy].data[:, 1]
        template_indices_i = (self.results.inav[ix, iy].data[:, 0]).astype('int16')
        orientations_i = orientations[template_indices_i]
        
        fig = plt.figure()
        ax = fig.add_subplot(111, projection="ipf", symmetry=phase.point_group)
        ax.scatter(orientations_i, c=correlations_i, cmap='inferno')

In [ ]:
sped_cropped = SPEDResult(*function2(function1(PATH_cropped)))
sped_mini = SPEDResult(*function2(function1(PATH_mini)))
sped_test = SPEDResult(*function2(function1(PATH_test)))

In [ ]:
def main_function(data_list):
    fig = data_list.single_phase.scatter(projection="ipf", return_figure=True)
    plt.show()

main_function(sped_test)

In [ ]:
def main_function2(data):
    
    reduced_zone = data.single_phase.map_into_symmetry_reduced_zone()
    
    rxyz = np.asarray(reduced_zone.to_axes_angles().xyz)
    rx = rxyz[0].ravel()
    ry = rxyz[1].ravel()
    rz = rxyz[2].ravel()
    
    pts = np.column_stack([rx, ry, rz])   # shape (N, 3)

    voxel = 0.02   # tolerance / bin size (choose based on your physics)
    
    # integer voxel indices
    keys = np.floor(pts / voxel).astype(np.int64)
    
    # unique voxels + counts
    uniq_keys, counts = np.unique(keys, axis=0, return_counts=True)
    
    # voxel centers for plotting
    centers = (uniq_keys + 0.5) * voxel
    xc, yc, zc = centers.T

    fig = go.Figure(
        go.Scatter3d(
            x=xc,
            y=yc,
            z=zc,
            mode="markers",
            marker=dict(
                size=5 + 5 * np.log1p(counts),   # scale size by density
                color=counts,                    # color = intensity
                colorscale="Viridis",
                colorbar=dict(title="Counts"),
                opacity=0.85
            )
        )
    )

    R = 3.4

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[-R, R], title="x"),
            yaxis=dict(range=[-R, R], title="y"),
            zaxis=dict(range=[-R, R], title="z"),
            aspectmode="cube"   # critical: equal scaling
        )
    )
    axis_lines = [
        # X-axis
        go.Scatter3d(
            x=[-R, R], y=[0, 0], z=[0, 0],
            mode="lines",
            line=dict(color="black", width=4),
            name="X axis",
            showlegend=False
        ),
        # Y-axis
        go.Scatter3d(
            x=[0, 0], y=[-R, R], z=[0, 0],
            mode="lines",
            line=dict(color="black", width=4),
            name="Y axis",
            showlegend=False
        ),
        # Z-axis
        go.Scatter3d(
            x=[0, 0], y=[0, 0], z=[-R, R],
            mode="lines",
            line=dict(color="black", width=4),
            name="Z axis",
            showlegend=False
        ),
    ]

    for trace in axis_lines:
        fig.add_trace(trace)

    fig.write_html(
    "Orientations_sped_cropped.html",
    include_plotlyjs="cdn"
    )

main_function2(sped_cropped)

# Misorientations

In [ ]:
def main_function3(data):

    difference = data.single_phase[5,5] - data.single_phase
    diff_red = difference.map_into_symmetry_reduced_zone()
    
    rxyz = np.asarray(diff_red.axis.xyz)

    r0x = rxyz[0].ravel()
    r0y = rxyz[1].ravel()
    r0z = rxyz[2].ravel()

    pts = np.column_stack([r0x, r0y, r0z])   # shape (N, 3)

    voxel = 0.02   # tolerance / bin size (choose based on your physics)
    
    # integer voxel indices
    keys = np.floor(pts / voxel).astype(np.int64)
    
    # unique voxels + counts
    uniq_keys, counts = np.unique(keys, axis=0, return_counts=True)
    
    # voxel centers for plotting
    centers = (uniq_keys + 0.5) * voxel
    xc, yc, zc = centers.T

    fig = go.Figure(
        go.Scatter3d(
            x=xc,
            y=yc,
            z=zc,
            mode="markers",
            marker=dict(
                size=5 + 5 * np.log1p(counts),   # scale size by density
                color=counts,                    # color = intensity
                colorscale="Viridis",
                colorbar=dict(title="Counts"),
                opacity=0.85
            )
        )
    )
    
    R = 3.4

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[-R, R], title="x"),
            yaxis=dict(range=[-R, R], title="y"),
            zaxis=dict(range=[-R, R], title="z"),
            aspectmode="cube"   # critical: equal scaling
        )
    )
    axis_lines = [
        # X-axis
        go.Scatter3d(
            x=[-R, R], y=[0, 0], z=[0, 0],
            mode="lines",
            line=dict(color="black", width=4),
            name="X axis",
            showlegend=False
        ),
        # Y-axis
        go.Scatter3d(
            x=[0, 0], y=[-R, R], z=[0, 0],
            mode="lines",
            line=dict(color="black", width=4),
            name="Y axis",
            showlegend=False
        ),
        # Z-axis
        go.Scatter3d(
            x=[0, 0], y=[0, 0], z=[-R, R],
            mode="lines",
            line=dict(color="black", width=4),
            name="Z axis",
            showlegend=False
        ),
    ]
    
    for trace in axis_lines:
        fig.add_trace(trace)
    
    fig.write_html(
    "Misorientations_sped_test.html",
    include_plotlyjs="cdn"
    )

main_function3(sped_test)

In [ ]:
phase.point_group.to_euler()

In [ ]:
def iterate_x(Orientations_list, x_max):
    pixel_misorientation_matrix = []
    pixel_misorientation_list = []
    for i in range(Orientations_list.shape[0]):
        # start a new row whenever we hit a multiple of x_max
        if i % x_max == 0 and i != 0:
            pixel_misorientation_matrix.append(pixel_misorientation_list)
            pixel_misorientation_list = []

        # if we are not at the last column in this row
        if i % x_max != x_max - 1 and i != Orientations_list.shape[0]:
            # if left, me, and right are identical orientations → put 0
            if (Orientations_list[i-1] == Orientations_list[i]
                and Orientations_list[i] == Orientations_list[i+1]):
                pixel_misorientation_list.append(0)
            else:
                # otherwise compute misorientation to pixel on the right
                pixel_misorientation_list.append(
                    Orientations_list[i].angle_with_outer(
                        Orientations_list[i+1], degrees=True
                    )[0][0]
                )
        else:
            # last pixel in the row, no right neighbour → 0
            pixel_misorientation_list.append(0)

    pixel_misorientation_matrix.append(pixel_misorientation_list)
    return pixel_misorientation_matrix

def iterate_y(Orientations_list, x_max, y_max, pixel_misorientation_matrix):
    for i in range(Orientations_list.shape[0]):
        if i + x_max < Orientations_list.shape[0]:
            # if at least one of (above, me, below) is different
            if (Orientations_list[i - x_max] != Orientations_list[i]
                or Orientations_list[i] != Orientations_list[i + x_max]):

                current_val = Orientations_list[i].angle_with_outer(
                    Orientations_list[i + x_max], degrees=True
                )[0][0]

                # pick the LARGEST of: "x-direction misori we already had"
                # vs "y-direction misori we just computed"
                if current_val > pixel_misorientation_matrix[i // x_max][i % x_max]:
                    pixel_misorientation_matrix[i // x_max][i % x_max] = current_val
    return pixel_misorientation_matrix

def grain_boundaries_misorientations(Orientations_list, x_max, y_max):
    pixel_misorientation_matrix = iterate_x(Orientations_list, x_max)
    pixel_misorientation_matrix = iterate_y(
        Orientations_list, x_max, y_max, pixel_misorientation_matrix
    )
    return np.array(pixel_misorientation_matrix)

In [ ]:
def iterate_x_updated(Orientations_list, x_max, ix, iy):
    pixel_misorientation_matrix = []
    pixel_misorientation_list = []
    for i in range(Orientations_list.shape[0]):
        # start a new row whenever we hit a multiple of x_max
        if i % x_max == 0 and i != 0:
            pixel_misorientation_matrix.append(pixel_misorientation_list)
            pixel_misorientation_list = []

        # if we are not at the last column in this row
        if i % x_max != x_max - 1 and i != Orientations_list.shape[0]:
            # if left, me, and right are identical orientations → put 0
            if (np.Orientations_list[i]  Orientations_list[ix + iy * x_max]):
                pixel_misorientation_list.append(0)
            else:
                # otherwise compute misorientation to pixel on the right
                pixel_misorientation_list.append(
                    Orientations_list[i].angle_with_outer(
                        Orientations_list[ix + iy * x_max], degrees=True
                    )[0][0]
                )
        else:
            # last pixel in the row, no right neighbour → 0
            pixel_misorientation_list.append(0)

    pixel_misorientation_matrix.append(pixel_misorientation_list)
    return pixel_misorientation_matrix

def iterate_y_updated(Orientations_list, x_max, y_max, pixel_misorientation_matrix, ix, iy):
    for i in range(Orientations_list.shape[0]):
        if i + x_max < Orientations_list.shape[0]:
            # if at least one of (above, me, below) is different
            if (Orientations_list[i - x_max] != Orientations_list[i]
                or Orientations_list[i] != Orientations_list[i + x_max]):

                current_val = Orientations_list[i].angle_with_outer(
                    Orientations_list[ix + iy * x_max], degrees=True
                )[0][0]

                # pick the LARGEST of: "x-direction misori we already had"
                # vs "y-direction misori we just computed"
                if current_val > pixel_misorientation_matrix[i // x_max][i % x_max]:
                    pixel_misorientation_matrix[i // x_max][i % x_max] = current_val
    return pixel_misorientation_matrix

def grain_boundaries_misorientations_updated(Orientations_list, x_max, y_max, ix, iy):
    pixel_misorientation_matrix = iterate_x_updated(Orientations_list, x_max, ix, iy)
    pixel_misorientation_matrix = iterate_y_updated(
        Orientations_list, x_max, y_max, pixel_misorientation_matrix, ix, iy
    )
    return np.array(pixel_misorientation_matrix)

In [ ]:
test = grain_boundaries_misorientations(SPED8_oris, 200, 100)

plt.figure(figsize=(6,5))
img = plt.imshow(test)
plt.colorbar(img)
plt.title("Misorientations")
plt.show()

In [ ]:
test = grain_boundaries_misorientations_updated(SPED8_oris, 200, 100, 180, 5)

plt.figure(figsize=(6,5))
img = plt.imshow(test)
plt.colorbar(img)
plt.scatter(180, 5, s=10, c='black', marker="o")
plt.title("Misorientations")
plt.show()

In [ ]:
from skimage.draw import line

fig, ax = plt.subplots()
ax.imshow(test, cmap="viridis")

coords = []   # <-- this will store the pixel indices, not float coords
clicks = []   # temporary storage for float input

def onclick(event):
    if event.inaxes != ax:
        return
    
    # get float click coordinates
    x, y = event.xdata, event.ydata
    clicks.append((x, y))

    # plot click marker
    ax.plot(x, y, 'rx')
    fig.canvas.draw()

    # when two points clicked
    if len(clicks) == 2:
        (x0, y0), (x1, y1) = clicks
        
        # convert to integer pixel indices
        r0, c0 = int(y0), int(x0)
        r1, c1 = int(y1), int(x1)

        # compute all pixel indices on the line
        rr, cc = line(r0, c0, r1, c1)

        # save them to coords instead of printing
        coords.extend(list(zip(rr, cc)))

        print("Saved pixel indices in coords:")
        print(coords)

        # draw line on figure
        ax.plot([x0, x1], [y0, y1], 'r-')
        fig.canvas.draw()

        # reset clicks for next line
        clicks.clear()

fig.canvas.mpl_connect('button_press_event', onclick)
plt.show()

In [ ]:
value = []

for j in coords:
    value.append(test[j])

x_array = np.arange(0, len(value))

plt.figure()
plt.plot(x_array, value)
plt.show()